# Kaggriculture Replay Lab — transformer chaque replay en avantage compétitif

**Plateforme :** [ouvrir Kaggriculture Replay Lab](https://kaggriculture-replay-lab-myeqmjdt6tnb4juseappyye.streamlit.app/)

Ce notebook explique pourquoi un score final ne suffit pas pour améliorer un agent Kaggriculture. Replay Lab transforme les fichiers JSON de match en diagnostics lisibles : trajectoire économique, volumes vendus, utilisation des unités, actions inutiles et valeur non encaissée.

> L'objectif n'est pas seulement de savoir **qui a gagné**, mais de comprendre **quand l'écart apparaît, par quel mécanisme, et quelle modification tester ensuite**.

## TL:DR

- Un leaderboard indique le résultat ; Replay Lab révèle le mécanisme de la performance.
- La comparaison quotidienne de l'argent localise le moment où une stratégie prend l'avantage.
- La ventilation des ventes distingue un problème de production, de logistique ou de timing de marché.
- Le taux de `PASS`, les actions et la valeur finale non encaissée signalent les pertes d'efficacité.
- Plusieurs replays peuvent être comparés dans un même portefeuille pour éviter de sur-optimiser l'agent contre un seul adversaire.
- Le diagnostic fournit une boucle reproductible : **observer → formuler une hypothèse → modifier → tester → conserver seulement les gains robustes**.

## 1. Le problème que résout la plateforme

Deux agents peuvent obtenir des scores très différents pour plusieurs raisons : montée en puissance trop lente, cultures mal choisies, travailleurs sous-utilisés, ventes tardives, stock oublié, ou réaction insuffisante à l'adversaire. Le score final mélange toutes ces causes.

Replay Lab sépare le match en cinq questions opérationnelles :

1. **Score** — quelle est la marge réelle ?
2. **Trajectoire** — quel jour l'écart se creuse-t-il ?
3. **Ventes** — quels produits expliquent le revenu ?
4. **Actions** — les unités créent-elles de la valeur ou attendent-elles ?
5. **Liquidation** — combien de valeur reste inutilisée à la fin ?

## 2. Démonstration reproductible

Le code ci-dessous reproduit le cœur analytique de Replay Lab sur les replays JSON disponibles dans `replays/ladder`. Il n'exécute aucun agent et ne modifie aucun replay.

In [ ]:
from pathlib import Path
from collections import Counter
import json
import tempfile
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "replays").exists() and (PROJECT_DIR / "kaggriculture" / "replays").exists():
    PROJECT_DIR = PROJECT_DIR / "kaggriculture"

REPLAY_FILES = sorted((PROJECT_DIR / "replays" / "ladder").glob("*.json"))
PRODUCTS = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK", "WOOL", "FERTILIZER"]
SEED_COSTS = {"WHEAT": 10, "CARROT": 20, "TOMATO": 50, "STRAWBERRY": 100, "MELON": 80}

def create_demo_replay():
    # Create a small synthetic replay when no external JSON is available.
    steps = []
    for day in range(30):
        farms = [
            {"money": 3000 + day * 1000, "tiles": [], "farmer": [4, 4], "hands": []},
            {"money": 3000 + day * 1300, "tiles": [], "farmer": [4, 4], "hands": []},
        ]
        records = []
        for player in range(2):
            product = "MELON" if player == 0 else "MILK"
            quantity = 3 + player * 2
            records.append({
                "observation": {
                    "day": day, "hour": 23, "player": player, "farms": farms,
                    "private": {"inventories": [], "seeds": {}, "shed": {}},
                    "market": {"prices": {item: 100 for item in PRODUCTS}},
                },
                "action": {
                    "farmer": ["PASS"] if day % (5 - player) == 0 else ["HARVEST"],
                    "hands": [], "market": [["SELL", product, quantity]],
                },
            })
        steps.append(records)
    return {"id": "synthetic-demo", "info": {"Agents": [{"Name": "Baseline"}, {"Name": "Improved agent"}]}, "steps": steps}

if not REPLAY_FILES:
    demo_path = Path(tempfile.gettempdir()) / "kaggriculture_replay_lab_demo.json"
    demo_path.write_text(json.dumps(create_demo_replay()), encoding="utf-8")
    REPLAY_FILES = [demo_path]
    print("Aucun replay externe trouvé : utilisation du replay synthétique intégré.")
else:
    print(f"{len(REPLAY_FILES)} replay(s) trouvé(s)")

In [ ]:
def analyse_replay(path):
    replay = json.loads(path.read_text(encoding="utf-8-sig"))
    steps = replay.get("steps") or []
    if not steps or not isinstance(steps[0], list):
        raise ValueError(f"Replay invalide : {path.name}")
    agents = (replay.get("info") or {}).get("Agents") or []
    players = []
    for p in range(len(steps[0])):
        name = agents[p].get("Name") if p < len(agents) and isinstance(agents[p], dict) else None
        unit_ops, market_qty, money_by_day = Counter(), Counter(), {}
        for step in steps:
            record = step[p] or {}; obs = record.get("observation") or {}; action = record.get("action") or {}
            farms = obs.get("farms") or []; farm = farms[p] if p < len(farms) else {}
            money_by_day[int(obs.get("day", 0) or 0)] = float(farm.get("money", 0) or 0)
            for command in [action.get("farmer", ["PASS"]), *(action.get("hands") or [])]:
                unit_ops[command[0] if command else "EMPTY"] += 1
            for order in action.get("market") or []:
                if order and len(order) >= 3 and isinstance(order[2], (int, float)):
                    market_qty[(order[0], order[1])] += int(order[2])
        last = steps[-1][p].get("observation") or {}; farms = last.get("farms") or []
        farm = farms[p] if p < len(farms) else {}; private = last.get("private") or {}
        prices = ((last.get("market") or {}).get("prices") or {})
        unharvested, weeds = Counter(), 0
        for row in farm.get("tiles") or []:
            for tile in row:
                if isinstance(tile, dict) and tile.get("kind") == "PLANT":
                    unharvested[tile.get("crop", "UNKNOWN")] += int(tile.get("yield_units", 0) or 0)
                elif isinstance(tile, dict) and tile.get("kind") == "WEED": weeds += 1
        carried = Counter()
        for inventory in private.get("inventories") or []: carried.update(inventory or {})
        seeds, shed = private.get("seeds") or {}, private.get("shed") or {}
        end_value = (sum(int(q or 0)*SEED_COSTS.get(k,0) for k,q in seeds.items())
                     + sum(int(q or 0)*int(prices.get(k,0) or 0) for k,q in unharvested.items())
                     + sum(int(q or 0)*int(prices.get(k,0) or 0) for k,q in carried.items())
                     + sum(int(q or 0)*int(prices.get(k,0) or 0) for k,q in shed.items()))
        total_actions = sum(unit_ops.values())
        players.append({"player":p,"agent":name or f"Joueur {p}","final_money":float(farm.get("money",0) or 0),
                        "money_by_day":money_by_day,"unit_ops":dict(unit_ops),"market_qty":dict(market_qty),
                        "pass_rate":unit_ops.get("PASS",0)/total_actions if total_actions else 0,
                        "end_value":end_value,"weeds":weeds})
    return {"episode":replay.get("id") or path.stem,"file":path.name,"players":players,"steps":len(steps)}

analyses = [analyse_replay(path) for path in REPLAY_FILES]
assert analyses, "Ajoutez au moins un replay JSON dans replays/ladder."
print("Analyse terminée sans erreur.")

## 3. Ce que Replay Lab rend immédiatement visible

In [ ]:
summary_rows = []
for match in analyses:
    ranked = sorted(match["players"], key=lambda x: x["final_money"], reverse=True)
    summary_rows.append({"Replay":match["file"], "Gagnant":ranked[0]["agent"],
                         "Score gagnant":ranked[0]["final_money"],
                         "Marge":ranked[0]["final_money"]-ranked[1]["final_money"],
                         "Valeur non encaissée du gagnant":ranked[0]["end_value"]})
summary = pd.DataFrame(summary_rows)
display(summary.style.format({"Score gagnant":"{:,.0f}","Marge":"{:+,.0f}","Valeur non encaissée du gagnant":"{:,.0f}"}))

### 3.1 Localiser le tournant économique

La courbe quotidienne distingue une mauvaise ouverture d'un mauvais final. Si l'écart apparaît tôt, on teste l'embauche, les achats et la production initiale. S'il apparaît tard, on inspecte plutôt les ventes et la liquidation.

In [ ]:
example = analyses[0]
fig, ax = plt.subplots(figsize=(10, 4.5))
for player in example["players"]:
    series = pd.Series(player["money_by_day"]).sort_index()
    ax.plot(series.index, series.values, marker="o", markersize=3, label=player["agent"])
ax.set(title=f"Trajectoire économique — {example['file']}", xlabel="Jour", ylabel="Argent")
ax.legend(); ax.grid(alpha=.25); plt.tight_layout(); plt.show()

### 3.2 Identifier le moteur de revenu

Comparer les volumes vendus répond à une question essentielle : l'adversaire gagne-t-il parce qu'il produit davantage, parce qu'il choisit un produit plus rentable, ou parce qu'il encaisse mieux son stock ?

In [ ]:
sales_rows = []
for player in example["players"]:
    for product in PRODUCTS:
        sales_rows.append({"Produit":product.title(),"Quantité":player["market_qty"].get(("SELL",product),0),"Agent":player["agent"]})
sales = pd.DataFrame(sales_rows)
pivot = sales.pivot(index="Produit", columns="Agent", values="Quantité").fillna(0)
display(pivot)
pivot.plot(kind="bar", figsize=(11,4.5), rot=35, title="Volumes vendus par produit")
plt.ylabel("Unités vendues"); plt.tight_layout(); plt.show()

### 3.3 Détecter l'inefficacité opérationnelle

`PASS` n'est pas automatiquement mauvais : il peut éviter une action non rentable. Mais un taux anormal, associé à une trajectoire faible, signale souvent une mauvaise coordination des travailleurs ou une stratégie trop rigide. La valeur non encaissée mesure quant à elle le coût approximatif du stock, des récoltes et des semences encore présents au dernier état observable.

In [ ]:
efficiency = pd.DataFrame([
    {"Replay":m["file"],"Agent":p["agent"],"Score final":p["final_money"],
     "PASS":p["pass_rate"],"Valeur non encaissée":p["end_value"],"Mauvaises herbes finales":p["weeds"]}
    for m in analyses for p in m["players"]
])
display(efficiency.style.format({"Score final":"{:,.0f}","PASS":"{:.1%}","Valeur non encaissée":"{:,.0f}"}))
assert efficiency["PASS"].between(0,1).all()
assert (efficiency[["Score final","Valeur non encaissée","Mauvaises herbes finales"]] >= 0).all().all()
print("Contrôles de cohérence réussis.")

## 4. Comment utiliser Replay Lab pour améliorer un agent

| Signal observé | Hypothèse à tester dans l'agent | Mesure de validation |
|---|---|---|
| Écart dès les premiers jours | ouverture ou embauche trop lente | argent et capacité de production aux jours 3, 5 et 10 |
| Faible volume d'un produit rentable | allocation de parcelles ou cycle de récolte insuffisant | volume vendu et marge finale |
| Beaucoup de `PASS` avec score faible | travailleurs bloqués ou calendrier trop rigide | taux de `PASS` et actions par unité |
| Forte valeur non encaissée | retour au hangar ou liquidation tardive | stock final et score après correction |
| Bon résultat contre un seul adversaire | sur-adaptation | taux de victoire sur plusieurs adversaires et deux positions |

### Boucle recommandée

1. Importer au minimum une défaite et une victoire représentative.
2. Localiser le premier jour où la trajectoire diverge.
3. Identifier un seul mécanisme candidat dans les ventes ou les actions.
4. Modifier une règle précise de l'agent.
5. Rejouer plusieurs graines dans les deux positions.
6. Conserver la modification seulement si elle améliore le portefeuille, pas un match isolé.

## 5. Pourquoi utiliser la plateforme plutôt qu'une lecture manuelle du JSON

- **Rapidité :** les centaines de tours sont résumés en indicateurs et graphiques.
- **Comparabilité :** les mêmes définitions sont appliquées à chaque agent et chaque replay.
- **Diagnostic actionnable :** chaque signal peut être relié à une règle de code à tester.
- **Réduction du surapprentissage :** l'onglet portefeuille permet d'étudier plusieurs adversaires.
- **Communication :** les graphiques rendent la stratégie compréhensible par des joueurs qui ne lisent pas le JSON.
- **Reproductibilité :** le résumé CSV et ce notebook permettent de conserver une trace des expériences.

Replay Lab ne garantit pas qu'une modification gagnera : il améliore la qualité des hypothèses et réduit le temps entre une défaite et une expérience utile.

## 6. Limites et bonnes pratiques

- Un replay est observationnel : une différence de ventes est associée à la victoire, mais ne prouve pas à elle seule la causalité.
- La valeur non encaissée est une estimation aux prix du dernier état observable.
- Les volumes ne tiennent pas à eux seuls compte du prix exact de chaque transaction.
- Il faut tester les changements sur plusieurs graines, adversaires et positions.
- Les replays analysés dans cette démonstration sont un petit échantillon local ; ils illustrent la méthode, pas une estimation universelle du leaderboard.

**Prochaine étape produit :** ajouter la comparaison automatique de versions, le prix réalisé moyen par produit et une détection du premier tournant économique.

## Conclusion

Kaggriculture Replay Lab transforme le replay en outil d'ingénierie. Sa valeur principale est de remplacer « mon agent a perdu » par une hypothèse mesurable comme « l'écart apparaît au jour 7, le rival vend davantage de lait et mon agent termine avec du stock ». Cette précision rend les itérations plus rapides, plus explicables et plus robustes.

➡️ [Tester Kaggriculture Replay Lab](https://kaggriculture-replay-lab-myeqmjdt6tnb4juseappyye.streamlit.app/)